# ECG Classification — Phase 2: Stage 1 Training (Normal vs Non-Normal)

This notebook trains the **Stage 1 binary classifier**: a TCN that takes a raw ECG window and predicts whether it is **Normal (0)** or **Non-Normal (1)**.

**Why TCN for Stage 1?**
From our experiments, TCN + ResNet1D with light augmentation gave Normal F1 = 0.88. The EfficientNet spectrogram model only gave Normal F1 = 0.83, and its confusion matrix showed 948 out of 5050 Normal samples misclassified as Other. Stage 1's only job is separating Normal — we use what the data proves is best at that.

**What this notebook does:**
- Loads all arrays saved by Phase 1
- Builds the TCN model as plain functions (no classes)
- Trains with RAdam, ReduceLROnPlateau, gradient clipping, label smoothing
- Prints per-epoch loss, accuracy, precision, recall, F1 for both train and val
- Detects and reports overfitting at every epoch
- Saves the best checkpoint and resumes automatically if the Colab session crashes
- Plots training curves at the end

## 1 · Imports

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.optim import RAdam
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import matplotlib.pyplot as plt

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

## 2 · Device and paths

In [ ]:
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = "checkpoints_stage1"
os.makedirs(CKPT_DIR, exist_ok=True)

CKPT_MODEL   = os.path.join(CKPT_DIR, "best_model.pt")
CKPT_STATE   = os.path.join(CKPT_DIR, "train_state.json")
HISTORY_PATH = os.path.join(CKPT_DIR, "history.json")

print("Device        :", DEVICE)
print("Checkpoint dir:", CKPT_DIR)

## 3 · Hyperparameters

In [ ]:
WINDOW_SIZE = 4500
BATCH_SIZE  = 64
LR          = 2e-4
WEIGHT_DECAY= 1e-4
MAX_EPOCHS  = 100
PATIENCE    = 15
VAL_FRAC    = 0.15
SEED        = 42

## 4 · Load data from Phase 1

In [ ]:
X_sig  = np.load("ECG_Hierarchical/X_train_sig.npy",    mmap_mode="r")
y_bin  = np.load("ECG_Hierarchical/y_train_binary.npy")

print("X_sig shape :", X_sig.shape)
print("y_bin shape :", y_bin.shape)
print("Class counts — 0 (Normal):", (y_bin == 0).sum(), "  1 (NonNormal):", (y_bin == 1).sum())

## 5 · Build DataLoaders

A fixed 85/15 train/val split at the window level. The test set is held out entirely and evaluated only in Phase 4.

In [ ]:
def make_loaders(X_sig, y_bin, val_frac, batch_size, seed):
    X_t = torch.tensor(np.array(X_sig), dtype=torch.float32)
    y_t = torch.tensor(y_bin,           dtype=torch.long)
    ds  = TensorDataset(X_t, y_t)

    n_val   = int(val_frac * len(ds))
    n_train = len(ds) - n_val
    gen     = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=gen)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    return train_loader, val_loader, n_train, n_val

In [ ]:
train_loader, val_loader, n_train, n_val = make_loaders(
    X_sig, y_bin, VAL_FRAC, BATCH_SIZE, SEED)

print("Train windows :", n_train)
print("Val windows   :", n_val)
print("Train batches :", len(train_loader))
print("Val batches   :", len(val_loader))

## 6 · Model — TCN as plain functions

No classes. Each building block is a plain function that returns an `nn.Module` built with `nn.Sequential` or a small `nn.Module` subclass only where PyTorch requires it (weight-normed convolutions need a forward pass, so two thin wrappers are unavoidable — kept as short as possible).

**Architecture decisions from experimental results:**
- Kernel size 9 (not 3) — larger receptive field per block, critical for capturing multi-beat patterns
- Expanding filters 1 → 128 → 256 → 512 → 1024 — more capacity at deeper layers
- Weight normalisation inside TCN (not batch norm) — stable with small batches on Colab
- PReLU (not ReLU) — prevents dead neurons in deep stacks
- SE (squeeze-excitation) attention — focuses on the most discriminative of the 1024 channels

In [ ]:
def se_block(channels, reduction=16):
    return nn.Sequential(
        nn.AdaptiveAvgPool1d(1),
        nn.Flatten(),
        nn.Linear(channels, channels // reduction),
        nn.ReLU(),
        nn.Linear(channels // reduction, channels),
        nn.Sigmoid()
    )

The SE block needs a custom forward (it multiplies the weight back onto the input). One minimal module — 6 lines.

In [ ]:
class SE1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gate = se_block(channels, reduction)
    def forward(self, x):
        w = self.gate(x).unsqueeze(-1)
        return x * w

In [ ]:
def temporal_block(in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.2):
    pad   = (kernel_size - 1) * dilation
    conv1 = nn.utils.weight_norm(
                nn.Conv1d(in_ch,  out_ch, kernel_size, dilation=dilation, padding=pad))
    conv2 = nn.utils.weight_norm(
                nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation, padding=pad))
    act1, act2   = nn.PReLU(), nn.PReLU()
    drop1, drop2 = nn.Dropout(dropout), nn.Dropout(dropout)
    proj         = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    return conv1, conv2, act1, act2, drop1, drop2, proj

The TCN block needs its own forward (causal trim + residual). One minimal module — 10 lines.

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=9, dilation=1, dropout=0.2):
        super().__init__()
        self.conv1, self.conv2, self.act1, self.act2,         self.drop1, self.drop2, self.proj =             temporal_block(in_ch, out_ch, kernel_size, dilation, dropout)
    def forward(self, x):
        out = self.drop1(self.act1(self.conv1(x)))
        out = self.drop2(self.act2(self.conv2(out)))
        out = out[:, :, :x.size(2)]
        return out + self.proj(x)

In [ ]:
def build_stage1_tcn(dropout=0.2):
    tcn = nn.Sequential(
        TCNBlock(1,    128,  dilation=1,  dropout=dropout),
        TCNBlock(128,  256,  dilation=2,  dropout=dropout),
        TCNBlock(256,  512,  dilation=4,  dropout=dropout),
        TCNBlock(512,  1024, dilation=8,  dropout=dropout),
    )
    se   = SE1D(1024)
    pool = nn.AdaptiveAvgPool1d(1)
    head = nn.Sequential(
        nn.Linear(1024, 256),
        nn.PReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 2)
    )
    model = nn.Sequential()
    model.add_module("tcn",  tcn)
    model.add_module("se",   se)
    model.add_module("pool", pool)
    model.add_module("head", head)
    return model

Sequential doesn't handle the reshape between pool and head, so the full forward is one small function:

In [ ]:
def forward_stage1(model, x):
    x   = x.unsqueeze(1)
    out = model.tcn(x)
    out = model.se(out)
    out = model.pool(out).squeeze(-1)
    return model.head(out)

## 7 · Instantiate model, optimiser, loss, scheduler

In [ ]:
model = build_stage1_tcn(dropout=0.2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total parameters    :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable:,}")

In [ ]:
optimizer = RAdam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

class_weights = torch.tensor([1.0, 2.5]).to(DEVICE)
criterion     = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-6)

print("Optimiser   :", optimizer.__class__.__name__)
print("Loss        :", criterion.__class__.__name__, " class weights:", class_weights.tolist())
print("Scheduler   : ReduceLROnPlateau  patience=8  factor=0.5")

## 8 · Checkpoint helpers

If the Colab session crashes, just re-run all cells. The training loop will:
1. Detect the saved checkpoint
2. Load model weights, optimiser state, and scheduler state
3. Resume from the exact epoch it stopped at

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch,
                    best_val_f1, patience_ctr, history):
    torch.save(model.state_dict(), CKPT_MODEL)
    state = {
        "epoch":        epoch,
        "best_val_f1":  best_val_f1,
        "patience_ctr": patience_ctr,
        "lr":           optimizer.param_groups[0]["lr"],
    }
    with open(CKPT_STATE, "w") as f:
        json.dump(state, f)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f)

In [ ]:
def load_checkpoint(model, optimizer, scheduler):
    if not os.path.exists(CKPT_MODEL) or not os.path.exists(CKPT_STATE):
        print("No checkpoint found. Starting from scratch.")
        return 0, 0.0, 0, {}
    model.load_state_dict(torch.load(CKPT_MODEL, map_location=DEVICE))
    with open(CKPT_STATE) as f:
        state = json.load(f)
    for pg in optimizer.param_groups:
        pg["lr"] = state["lr"]
    history = {}
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)
    print(f"Resumed from epoch {state['epoch']+1}  best val F1: {state['best_val_f1']:.4f}")
    return state["epoch"] + 1, state["best_val_f1"], state["patience_ctr"], history

## 9 · Per-epoch metrics helpers

Every epoch prints: loss, accuracy, precision, recall, F1 for both train and val.
Overfitting is flagged when train F1 exceeds val F1 by more than 0.08.

In [ ]:
def compute_metrics(trues, preds, losses):
    loss  = np.mean(losses)
    acc   = np.mean(np.array(trues) == np.array(preds))
    prec  = precision_score(trues, preds, average="macro", zero_division=0)
    rec   = recall_score(trues, preds, average="macro", zero_division=0)
    f1    = f1_score(trues, preds, average="macro", zero_division=0)
    f1_0  = f1_score(trues, preds, labels=[0], average="macro", zero_division=0)
    f1_1  = f1_score(trues, preds, labels=[1], average="macro", zero_division=0)
    return {"loss": loss, "acc": acc, "prec": prec, "rec": rec,
            "f1": f1, "f1_N": f1_0, "f1_NN": f1_1}

In [ ]:
def print_epoch(epoch, total, tr, vl, lr, overfit):
    flag = "  [OVERFIT]" if overfit else ""
    print(
        f"Epoch {epoch:>3}/{total}"
        f"  lr={lr:.2e}"
        f"  | TRAIN  loss={tr['loss']:.4f}  acc={tr['acc']:.4f}"
        f"  prec={tr['prec']:.4f}  rec={tr['rec']:.4f}"
        f"  F1={tr['f1']:.4f}  F1_N={tr['f1_N']:.4f}  F1_NN={tr['f1_NN']:.4f}"
        f"  | VAL  loss={vl['loss']:.4f}  acc={vl['acc']:.4f}"
        f"  prec={vl['prec']:.4f}  rec={vl['rec']:.4f}"
        f"  F1={vl['f1']:.4f}  F1_N={vl['f1_N']:.4f}  F1_NN={vl['f1_NN']:.4f}"
        f"{flag}"
    )

## 10 · One-epoch train and eval functions

In [ ]:
def run_train_epoch(model, loader, optimizer, criterion):
    model.train()
    all_preds, all_trues, all_losses = [], [], []
    for x_sig, y in loader:
        x_sig, y = x_sig.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = forward_stage1(model, x_sig)
        loss   = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_trues.extend(y.cpu().numpy())
        all_losses.append(loss.item())
    return compute_metrics(all_trues, all_preds, all_losses)

In [ ]:
def run_val_epoch(model, loader, criterion):
    model.eval()
    all_preds, all_trues, all_losses = [], [], []
    with torch.no_grad():
        for x_sig, y in loader:
            x_sig, y = x_sig.to(DEVICE), y.to(DEVICE)
            logits   = forward_stage1(model, x_sig)
            loss     = criterion(logits, y)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_trues.extend(y.cpu().numpy())
            all_losses.append(loss.item())
    return compute_metrics(all_trues, all_preds, all_losses)

## 11 · Training loop

Resumes automatically from the last saved checkpoint.
Saves a new checkpoint after every epoch where val F1 improves.
Early stopping triggers after `PATIENCE` epochs with no improvement.

In [ ]:
start_epoch, best_val_f1, patience_ctr, history = load_checkpoint(
    model, optimizer, scheduler)

if not history:
    history = {"train": [], "val": []}

print(f"Starting from epoch {start_epoch + 1}")
print(f"Max epochs: {MAX_EPOCHS}  Early stopping patience: {PATIENCE}")

In [ ]:
for epoch in range(start_epoch, MAX_EPOCHS):

    tr_metrics = run_train_epoch(model, train_loader, optimizer, criterion)
    vl_metrics = run_val_epoch(model, val_loader, criterion)

    lr       = optimizer.param_groups[0]["lr"]
    overfit  = (tr_metrics["f1"] - vl_metrics["f1"]) > 0.08
    scheduler.step(vl_metrics["f1"])

    history["train"].append(tr_metrics)
    history["val"].append(vl_metrics)

    print_epoch(epoch + 1, MAX_EPOCHS, tr_metrics, vl_metrics, lr, overfit)

    if vl_metrics["f1"] > best_val_f1:
        best_val_f1  = vl_metrics["f1"]
        patience_ctr = 0
        save_checkpoint(model, optimizer, scheduler,
                        epoch, best_val_f1, patience_ctr, history)
        print(f"  Checkpoint saved  best val F1: {best_val_f1:.4f}")
    else:
        patience_ctr += 1
        save_checkpoint(model, optimizer, scheduler,
                        epoch, best_val_f1, patience_ctr, history)
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}  "
                  f"no improvement for {PATIENCE} epochs")
            break

print(f"Training complete.  Best val macro F1: {best_val_f1:.4f}")

## 12 · Training curves

In [ ]:
def plot_history(history, save_path):
    tr = history["train"]
    vl = history["val"]
    epochs = range(1, len(tr) + 1)

    keys   = ["loss", "acc", "f1", "prec", "rec"]
    titles = ["Loss", "Accuracy", "Macro F1", "Macro Precision", "Macro Recall"]

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    for ax, key, title in zip(axes, keys, titles):
        ax.plot(epochs, [m[key] for m in tr], label="train")
        ax.plot(epochs, [m[key] for m in vl], label="val")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.show()
    print("Curve saved:", save_path)

In [ ]:
with open(HISTORY_PATH) as f:
    history = json.load(f)

plot_history(history, os.path.join(CKPT_DIR, "training_curves.png"))

## 13 · Per-class F1 over epochs — N vs NonNormal

In [ ]:
epochs = range(1, len(history["train"]) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, [m["f1_N"]  for m in history["train"]], label="train F1_Normal")
ax.plot(epochs, [m["f1_N"]  for m in history["val"]],   label="val F1_Normal")
ax.plot(epochs, [m["f1_NN"] for m in history["train"]], label="train F1_NonNormal", linestyle="--")
ax.plot(epochs, [m["f1_NN"] for m in history["val"]],   label="val F1_NonNormal",   linestyle="--")
ax.set_title("Per-class F1 over epochs")
ax.set_xlabel("Epoch")
ax.set_ylabel("F1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "per_class_f1.png"), dpi=100)
plt.show()

## 14 · Overfitting gap over epochs

In [ ]:
gap    = [t["f1"] - v["f1"]
          for t, v in zip(history["train"], history["val"])]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(epochs, gap, color="tomato")
ax.axhline(0.08, color="gray", linestyle="--", label="overfit threshold (0.08)")
ax.set_title("Train - Val F1 gap (overfitting indicator)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Gap")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "overfit_gap.png"), dpi=100)
plt.show()

max_gap = max(gap)
print(f"Maximum overfitting gap: {max_gap:.4f}"
      f"  {'(overfitting occurred)' if max_gap > 0.08 else '(no significant overfitting)'}")

## 15 · Final evaluation on the validation split using the best checkpoint

In [ ]:
model.load_state_dict(torch.load(CKPT_MODEL, map_location=DEVICE))
model.eval()

all_preds, all_trues, all_probs = [], [], []
with torch.no_grad():
    for x_sig, y in val_loader:
        logits = forward_stage1(model, x_sig.to(DEVICE))
        probs  = F.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_trues.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_trues  = np.array(all_trues)
all_probs  = np.array(all_probs)

In [ ]:
print("Classification report (best checkpoint, val split):")
print(classification_report(all_trues, all_preds,
      target_names=["Normal (0)", "NonNormal (1)"], digits=4))

macro_f1 = f1_score(all_trues, all_preds, average="macro")
print(f"Val macro F1: {macro_f1:.4f}")
if macro_f1 >= 0.92:
    print("Target met (>= 0.92). Proceed to Phase 3.")
elif macro_f1 >= 0.88:
    print("Acceptable. Consider tuning class weights before Phase 3.")
else:
    print("Below target. Check class weights, augmentation, and label correctness.")

## 16 · Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(all_trues, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Normal", "NonNormal"])
ax.set_yticklabels(["Normal", "NonNormal"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Stage 1 confusion matrix (val split, best checkpoint)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

## 17 · Stage 1 threshold search

The default threshold is 0.5 (argmax). We sweep the NonNormal probability threshold on the validation split to find the value that maximises macro F1. This tuned threshold is used in Phase 4 at inference.

In [ ]:
def sweep_threshold(probs, trues, thresholds):
    results = []
    for thr in thresholds:
        preds = (probs[:, 1] >= thr).astype(int)
        f1    = f1_score(trues, preds, average="macro", zero_division=0)
        prec  = precision_score(trues, preds, average="macro", zero_division=0)
        rec   = recall_score(trues, preds, average="macro", zero_division=0)
        results.append({"threshold": thr, "f1": f1, "prec": prec, "rec": rec})
    return results

In [ ]:
thresholds = np.arange(0.30, 0.71, 0.05).round(2)
thr_results = sweep_threshold(all_probs, all_trues, thresholds)

print(f"{'Threshold':>10}  {'Macro F1':>10}  {'Precision':>10}  {'Recall':>10}")
for r in thr_results:
    marker = "  <-- best" if r["f1"] == max(x["f1"] for x in thr_results) else ""
    print(f"{r['threshold']:>10.2f}  {r['f1']:>10.4f}  {r['prec']:>10.4f}  {r['rec']:>10.4f}{marker}")

In [ ]:
best_thr = max(thr_results, key=lambda x: x["f1"])["threshold"]
print(f"Best threshold on val split: {best_thr}")

import json
with open(os.path.join(CKPT_DIR, "best_threshold.json"), "w") as f:
    json.dump({"stage1_threshold": float(best_thr)}, f)

print("Saved to", os.path.join(CKPT_DIR, "best_threshold.json"))

## 18 · Summary

What is saved in `checkpoints_stage1/` for use in Phase 3 and Phase 4:

| File | Contents |
|------|----------|
| `best_model.pt` | Best model weights (highest val macro F1) |
| `train_state.json` | Epoch, best F1, patience counter, LR at time of save |
| `history.json` | Full per-epoch metrics for both train and val |
| `best_threshold.json` | Tuned NonNormal probability threshold |
| `training_curves.png` | Loss / Acc / F1 / Prec / Rec over epochs |
| `per_class_f1.png` | Normal vs NonNormal F1 over epochs |
| `overfit_gap.png` | Train minus Val F1 gap |
| `confusion_matrix.png` | Best-checkpoint confusion matrix on val |

In [ ]:
print("Phase 2 complete.")
print(f"Best val macro F1 : {best_val_f1:.4f}")
print(f"Best threshold    : {best_thr}")
print("Proceed to Phase 3 — Stage 2 (AF vs Other vs Noisy).")